# 03 · Evaluación — FID, KID, IS y exactitud por órgano

**Qué mide cada cosa**
| Métrica | Mide | Cómo leerla |
|---|---|---|
| **FID** (↓) | distancia entre características Inception de reales y generadas | con pocas imágenes está **sesgada hacia arriba**: compárala contra el *piso* |
| **KID** (↓) | parecida al FID, sin sesgo con pocas muestras | la más fiable aquí |
| **IS** (↑) | nitidez/diversidad según un clasificador de ImageNet | poco informativa en histología; compárala con la de las reales |
| **Exactitud del clasificador** (↑) | un ResNet18 entrenado con PanNuke real ¿reconoce el órgano pedido? | mide si el condicionamiento **funciona** |
| **Piso (real vs real)** | FID/KID entre dos conjuntos de imágenes **reales** | el mejor valor alcanzable con este N |

**Barrido de CFG:** la guía *classifier-free* (`cfg`) controla cuánto obedece el modelo al órgano pedido. Muy bajo = ignora el órgano; muy alto = imágenes saturadas y poco variadas. El óptimo de ImageNet (≈29) **no** aplica a 3 clases, por eso se calibra aquí.

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/fractal_histopatologia_256/code")   # carpeta con fg_*.py

!pip -q install "timm==0.9.12" "torchmetrics[image]" torch-fidelity scikit-learn pandas
import json, numpy as np, pandas as pd, torch
import fg_config as cfg, fg_common as fg, fg_eval as fe
fg.mount_drive(); fg.setup_repo(); fg.seed_everything(); fg.gpu_info()
X, y, classes, tr_idx, va_idx = fg.load_clean_data()
model, cl_m = fg.load_for_inference("best_fp16.pth")      # cambia a "final_fp16.pth" si quieres comparar
assert cl_m == classes
K = len(classes); print(classes)

## 1. Clasificador de órganos (con imágenes reales)

In [ ]:
clf = fe.train_classifier(X, y, tr_idx, va_idx, classes, epochs=5)
print("Exactitud en reales (validación):", round(fe.real_accuracy(clf, X, y, va_idx), 3))

## 2. Barrido de CFG (pocas imágenes, para elegir)
Ajusta `SWEEP` y `N_SWEEP` según el tiempo que viste por imagen (la generación a 256 px es lenta). Es reanudable: salta lo que ya existe en Drive.

In [ ]:
SWEEP, N_SWEEP = [1.0, 3.0, 6.0], 4
rows, grids = [], []
for c in SWEEP:
    d = f"{cfg.RES_DIR}/barrido_cfg/cfg{c}"
    fg.generate_to_disk(model, classes, N_SWEEP, d, bs=4, seed_base=777, cfg_scale=c)
    gen = {o: fe.load_png_folder(f"{d}/{o}") for o in classes}
    acc, prob = fe.gen_accuracy(clf, gen, classes)
    rows.append({"cfg": c, "exactitud": np.mean(list(acc.values())), "P(órgano) medio": np.mean(list(prob.values()))})
    grids += [gen[o][0].float() / 255 for o in classes]
display(pd.DataFrame(rows).round(3))
fe.show_grid(grids, [f"cfg {c} · {o}" for c in SWEEP for o in classes], ncols=K, size=3.0)

In [ ]:
# Elige el CFG mirando la tabla Y las imágenes (no solo el número) y guárdalo:
BEST_CFG = 3.0
cfg.CFG = BEST_CFG
json.dump({"cfg": BEST_CFG, "num_iter_list": list(cfg.NUM_ITER_LIST), "temperature": cfg.TEMPERATURE},
          open(f"{cfg.RES_DIR}/config_gen.json", "w"))

## 3. Generación para métricas (reanudable)

In [ ]:
N_PER_CLASS = 32          # sube a 64-100 si el tiempo lo permite: más imágenes = métricas más estables
gdir = f"{cfg.RES_DIR}/generadas/cfg{cfg.CFG}"
fg.generate_to_disk(model, classes, N_PER_CLASS, gdir, bs=4, seed_base=2024, cfg_scale=cfg.CFG)

## 4. Métricas globales y por órgano

In [ ]:
gen = {o: fe.load_png_folder(f"{gdir}/{o}") for o in classes}
ref = fe.real_sample(X, y, tr_idx, 200, K)                        # referencia: reales de entrenamiento
flo = fe.real_sample(X, y, va_idx, N_PER_CLASS, K, seed=1)        # piso: reales de validación (mismo N)
cat = lambda d: torch.cat([d[k] for k in sorted(d)])
gen_all = torch.cat([gen[o] for o in classes])

M = fe.distribution_metrics(cat(ref), gen_all)
F = fe.distribution_metrics(cat(ref), cat(flo), with_is=False)
acc, prob = fe.gen_accuracy(clf, gen, classes)
acc_real = fe.real_accuracy(clf, X, y, va_idx)

per = []
for ci, o in enumerate(classes):
    g = fe.distribution_metrics(ref[ci], gen[o], with_is=False)
    f = fe.distribution_metrics(ref[ci], flo[ci], with_is=False)
    per.append({"órgano": o, "FID gen": g["FID"], "FID piso": f["FID"], "KID gen": g["KID"], "KID piso": f["KID"],
                "exactitud": acc[o], "P(órgano)": prob[o]})
df_per = pd.DataFrame(per).round(4)

RES = {"cfg": cfg.CFG, "n_por_organo": N_PER_CLASS, "FID": M["FID"], "FID_piso": F["FID"], "KID": M["KID"],
       "KID_piso": F["KID"], "IS": M["IS"], "exactitud_gen": float(np.mean(list(acc.values()))),
       "exactitud_real": acc_real}
json.dump(RES, open(f"{cfg.RES_DIR}/metricas_globales.json", "w"), indent=2)
df_per.to_csv(f"{cfg.RES_DIR}/metricas_por_organo.csv", index=False)
print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in RES.items()}, indent=2, ensure_ascii=False))
display(df_per)

**Cómo interpretarlo:** compara cada métrica con su *piso*. Si FID gen ≈ FID piso, el modelo es casi indistinguible de datos reales con este N. Si la exactitud de generadas es cercana a la de reales, el condicionamiento por órgano funciona.

In [ ]:
# Figura para el informe: reales (fila 1 de cada par) vs generadas
imgs, tit = [], []
for ci, o in enumerate(classes):
    r = torch.from_numpy(X[va_idx[y[va_idx] == ci][:6]]).permute(0, 3, 1, 2).float() / 255
    imgs += list(r) + list(gen[o][:6].float() / 255)
    tit += [f"{o} real"] * 6 + [f"{o} generada"] * 6
fe.show_grid(imgs, tit, ncols=6, path=f"{cfg.RES_DIR}/reales_vs_generadas.png", size=2.2)